# Notebook 03 — Model Training (leakage-free pipelines)

### What this notebook does
Trains the same six classifiers as before — Logistic Regression, Decision Tree, Random Forest, XGBoost, LightGBM
and a Stacking Ensemble — but every learned step now lives **inside one scikit-learn Pipeline**:

`raw features → fill remaining blanks (+ "was missing" flags) → encode categories → (scale, LR only) → model`

Because the whole pipeline is fitted on the **training set only**, and re-fitted inside every cross-validation fold,
no information from validation or test respondents can reach the model (reviewer comments #1, #5).

### What changed from the old Notebooks 04–08
| Old | New | Reviewer comment |
|---|---|---|
| Imputation fitted on the full dataset before the split | Imputation inside the pipeline, fitted on training folds only | #1 |
| SMOTE applied **before** cross-validation | **No SMOTE** — class weights instead (no synthetic people, no fractional yes/no values) | #5, #7 |
| Nominal codes treated as numbers (e.g. marital status 2.4) | One-hot encoding | #7 |
| Validation set computed but unused | Validation set kept for calibration and threshold choice (Notebook 04) | #5 |
| Final hyperparameters not reported | Saved for every model | #5, #13 |

The **test set is not used at all** in this notebook.

**Input** — `data/processed/brfss_model_ready_v2.csv`, `outputs/feature_config_v2.json`
**Output** — `models/v2/*.joblib`, `outputs/v2_tuning_results.csv`, `outputs/v2_best_params.json`, `outputs/v2_validation_auc.csv`

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', 'lightgbm', '-q'])

import pandas as pd
import numpy as np
import os, json, time, joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.base import clone
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import sklearn, xgboost, lightgbm

print(f"scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__} | lightgbm {lightgbm.__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR     = '/content/drive/MyDrive/stroke_prediction'
READY_PATH   = os.path.join(BASE_DIR, 'data', 'processed', 'brfss_model_ready_v2.csv')
CONFIG_PATH  = os.path.join(BASE_DIR, 'outputs', 'feature_config_v2.json')
MODELS_DIR   = os.path.join(BASE_DIR, 'models', 'v2')
OUTPUTS_DIR  = os.path.join(BASE_DIR, 'outputs')
TUNING_PATH  = os.path.join(OUTPUTS_DIR, 'v2_tuning_results.csv')
PARAMS_PATH  = os.path.join(OUTPUTS_DIR, 'v2_best_params.json')
VAL_PATH     = os.path.join(OUTPUTS_DIR, 'v2_validation_auc.csv')
os.makedirs(MODELS_DIR, exist_ok=True)

# ── Run settings ──────────────────────────────────────────────────────────
FEATURE_SETS = ['A', 'B']      # A = all core features, B = without likely post-stroke features
SAMPLE       = 'primary'       # 'primary' = partial interviews excluded; 'all' = sensitivity analysis
TUNE_N       = 100_000         # tuning runs on a stratified subsample of the TRAINING set (speed)
CV_FOLDS     = 3
SEED         = 42

config = json.load(open(CONFIG_PATH))
df = pd.read_csv(READY_PATH)
if SAMPLE == 'primary':
    df = df[df['meta_keep_primary']]

train = df[df['split'] == 'train']
val   = df[df['split'] == 'validation']
print(f"Sample: {SAMPLE} | train {len(train):,} (stroke {train['stroke'].mean()*100:.2f}%) | "
      f"validation {len(val):,} | test not loaded into any model")

### Step 1 — The preprocessing pipeline

| Feature type | Blanks filled with | Then |
|---|---|---|
| continuous, ordinal | training-set **median** + a "was missing" 0/1 column | scaled (Logistic Regression only) |
| binary | training-set **most frequent** value + a "was missing" column | — |
| nominal | its own **"missing" category** | one-hot encoded |

"Training-set" means the training part of whatever is being fitted — inside cross-validation, only that fold's
training data. This is what makes the pipeline leakage-free.

In [ ]:
def make_preprocessor(features, scale):
    num = [f for f in config['continuous'] + config['ordinal'] if f in features]
    binr = [f for f in config['binary'] if f in features]
    nom = [f for f in config['nominal'] if f in features]

    num_steps = [('impute', SimpleImputer(strategy='median', add_indicator=True))]
    if scale:
        num_steps.append(('scale', StandardScaler()))
    return ColumnTransformer([
        ('num', Pipeline(num_steps), num),
        ('bin', SimpleImputer(strategy='most_frequent', add_indicator=True), binr),
        ('nom', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value=-1)),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), nom),
    ])

neg, pos = (train['stroke'] == 0).sum(), (train['stroke'] == 1).sum()
POS_WEIGHT = neg / pos   # used by XGBoost (same effect as class_weight='balanced')
print(f"Class weight for stroke cases: {POS_WEIGHT:.1f}  (non-stroke {neg:,} / stroke {pos:,})")

### Step 2 — Models and search spaces

Imbalance is handled with **class weights**: each stroke case counts about 22× more than a non-stroke case
during training. Probabilities from weighted models are not real-world risks — they are calibrated on the validation
set in Notebook 04 (reviewer comment #4).

Hyperparameters are tuned by randomised search with 3-fold stratified cross-validation, maximising AUC-ROC, on a
100,000-respondent stratified subsample **of the training set**. The best settings are then refitted on the full
training set. Average precision (PR-AUC) is recorded alongside.

In [ ]:
def model_specs():
    return {
        'LogisticRegression': dict(
            scale=True, n_iter=10,
            model=LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=2000, random_state=SEED),
            grid={'model__C': [0.001, 0.01, 0.1, 1, 10], 'model__penalty': ['l1', 'l2']}),
        'DecisionTree': dict(
            scale=False, n_iter=15,
            model=DecisionTreeClassifier(class_weight='balanced', random_state=SEED),
            grid={'model__max_depth': [4, 6, 8, 10, 12],
                  'model__min_samples_leaf': [20, 50, 100, 200],
                  'model__criterion': ['gini', 'entropy']}),
        'RandomForest': dict(
            scale=False, n_iter=10,
            model=RandomForestClassifier(class_weight='balanced_subsample', n_jobs=-1, random_state=SEED),
            grid={'model__n_estimators': [200, 300],
                  'model__max_depth': [12, 20, None],
                  'model__min_samples_leaf': [5, 20, 50],
                  'model__max_features': ['sqrt', 0.3],
                  'model__max_samples': [0.5, 0.8]}),
        'XGBoost': dict(
            scale=False, n_iter=15,
            model=XGBClassifier(scale_pos_weight=POS_WEIGHT, tree_method='hist', eval_metric='auc',
                                n_jobs=-1, random_state=SEED, verbosity=0),
            grid={'model__n_estimators': [200, 400, 600],
                  'model__max_depth': [3, 4, 6],
                  'model__learning_rate': [0.03, 0.05, 0.1],
                  'model__subsample': [0.7, 0.9],
                  'model__colsample_bytree': [0.7, 0.9],
                  'model__min_child_weight': [1, 5, 10]}),
        'LightGBM': dict(
            scale=False, n_iter=15,
            model=LGBMClassifier(class_weight='balanced', subsample_freq=1, n_jobs=-1,
                                 random_state=SEED, verbose=-1),
            grid={'model__n_estimators': [200, 400, 600],
                  'model__num_leaves': [15, 31, 63],
                  'model__learning_rate': [0.03, 0.05, 0.1],
                  'model__min_child_samples': [20, 50, 100],
                  'model__subsample': [0.7, 0.9],
                  'model__colsample_bytree': [0.7, 0.9]}),
    }

for name, s in model_specs().items():
    print(f"{name:<20} search: {s['n_iter']} settings x {CV_FOLDS} folds")

### Step 3 — Tune, refit and save every model

Each finished model is saved immediately. If Colab disconnects, just re-run the notebook: models that already
exist are loaded instead of retrained.

In [ ]:
def load_json(path, default):
    return json.load(open(path)) if os.path.exists(path) else default

best_params = load_json(PARAMS_PATH, {})
tuning_rows = pd.read_csv(TUNING_PATH).to_dict('records') if os.path.exists(TUNING_PATH) else []
fitted      = {}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

for fs in FEATURE_SETS:
    features = config[f'features_{fs}']
    X_train, y_train = train[features], train['stroke']
    X_tune, _, y_tune, _ = train_test_split(X_train, y_train, train_size=min(TUNE_N, len(X_train) - 1),
                                            stratify=y_train, random_state=SEED)
    print(f"\n{'='*70}\nFeature set {fs}: {len(features)} features | tuning on {len(X_tune):,} | "
          f"refit on {len(X_train):,}\n{'='*70}")

    for name, spec in model_specs().items():
        key  = f'{SAMPLE}_{fs}_{name}'
        path = os.path.join(MODELS_DIR, f'{key}.joblib')
        if os.path.exists(path):
            fitted[key] = joblib.load(path)
            print(f"  {name:<20} loaded from disk")
            continue

        pipe = Pipeline([('prep', make_preprocessor(features, spec['scale'])), ('model', spec['model'])])
        start = time.time()
        search = RandomizedSearchCV(pipe, spec['grid'], n_iter=spec['n_iter'], cv=cv,
                                    scoring={'auc': 'roc_auc', 'ap': 'average_precision'}, refit='auc',
                                    random_state=SEED, n_jobs=1)
        search.fit(X_tune, y_tune)
        best = clone(pipe).set_params(**search.best_params_).fit(X_train, y_train)
        minutes = (time.time() - start) / 60

        joblib.dump(best, path)
        fitted[key] = best
        best_params[key] = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
        i = search.best_index_
        tuning_rows = [r for r in tuning_rows if r['key'] != key]   # a retrained model replaces its old row
        tuning_rows.append({'key': key, 'feature_set': fs, 'model': name,
                            'cv_auc': round(search.cv_results_['mean_test_auc'][i], 4),
                            'cv_auc_sd': round(search.cv_results_['std_test_auc'][i], 4),
                            'cv_ap': round(search.cv_results_['mean_test_ap'][i], 4),
                            'minutes': round(minutes, 1)})
        json.dump(best_params, open(PARAMS_PATH, 'w'), indent=2, default=str)
        pd.DataFrame(tuning_rows).to_csv(TUNING_PATH, index=False)
        print(f"  {name:<20} CV AUC {tuning_rows[-1]['cv_auc']:.4f} ± {tuning_rows[-1]['cv_auc_sd']:.4f} "
              f"| {minutes:.1f} min | {best_params[key]}")

### Step 4 — Stacking Ensemble

Random Forest, XGBoost and LightGBM (with their tuned settings) are combined by a Logistic Regression that learns
how much to trust each one. The combiner is trained on **out-of-fold** predictions (3-fold, training set only), so
it never sees a prediction a base model made on its own training data.

In [ ]:
for fs in FEATURE_SETS:
    key  = f'{SAMPLE}_{fs}_Stacking'
    path = os.path.join(MODELS_DIR, f'{key}.joblib')
    if os.path.exists(path):
        fitted[key] = joblib.load(path)
        print(f"Feature set {fs}: Stacking loaded from disk")
        continue

    features = config[f'features_{fs}']
    base = [(n.lower(), clone(fitted[f'{SAMPLE}_{fs}_{n}'])) for n in ['RandomForest', 'XGBoost', 'LightGBM']]
    stack = StackingClassifier(estimators=base,
                               final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
                               cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED),
                               stack_method='predict_proba', n_jobs=1)
    start = time.time()
    stack.fit(train[features], train['stroke'])
    joblib.dump(stack, path)
    fitted[key] = stack
    print(f"Feature set {fs}: Stacking fitted in {(time.time()-start)/60:.1f} min | "
          f"combiner weights {np.round(stack.final_estimator_.coef_[0], 3).tolist()}")

### Step 5 — Validation AUC (model comparison only)

AUC-ROC and average precision on the **validation set**, using uncalibrated scores. These are for comparing models
and choosing which ones go forward; they are **not** the final results. Final results — with calibration,
thresholds and confidence intervals — come from the test set in Notebook 04.

In [ ]:
rows = []
for fs in FEATURE_SETS:
    features = config[f'features_{fs}']
    for name in ['LogisticRegression', 'DecisionTree', 'RandomForest', 'XGBoost', 'LightGBM', 'Stacking']:
        key = f'{SAMPLE}_{fs}_{name}'
        p = fitted[key].predict_proba(val[features])[:, 1]
        rows.append({'feature_set': fs, 'model': name,
                     'val_auc': round(roc_auc_score(val['stroke'], p), 4),
                     'val_ap' : round(average_precision_score(val['stroke'], p), 4)})

val_df = pd.DataFrame(rows)
val_df.to_csv(VAL_PATH, index=False)
print(val_df.pivot(index='model', columns='feature_set', values=['val_auc', 'val_ap']).round(4).to_string())
print(f"\nStroke prevalence in validation set: {val['stroke'].mean():.4f}  "
      f"(= average precision of a random guess)")

print("\n" + "=" * 60)
print("  Notebook 03 complete")
print("=" * 60)
print(f"  Models saved   : {MODELS_DIR}")
print(f"  Tuning results : {TUNING_PATH}")
print(f"  Best params    : {PARAMS_PATH}")
print(f"  Validation AUC : {VAL_PATH}")
print("=" * 60)